# Notebook 2 — Inter-Annotator Agreement (κ) + Gemini Zero-Shot Baseline

Run this **after** two annotators have filled `annotator_1_label` and `annotator_2_label` in
`annotation_sample_BLANK.xlsx`.

It does two independent things and writes both into `outputs/paper_q1/tables/`:

**Part A — Inter-annotator agreement (no API, no GPU, seconds).**
- Cohen's κ between the two annotators.
- Raw percent agreement, per-class agreement, and each annotator vs. the consolidated gold label.
- Writes `table_iaa_kappa.tex` + `.csv`.

**Part B — Gemini 2.5 Flash zero-shot baseline (API, no GPU).**
- Classifies a test sample zero-shot into the five classes and scores Macro-F1 / Weighted-F1 /
  Accuracy against gold.
- Writes `table_llm_baseline.tex` + `.csv`.
- Cost/time controls are in the config cell (subset size, model id, rate limit).

Run Part A first; it is free and is the higher-priority result. Part B is optional and billed per call.


In [ ]:
#pip install google-generativeai openpyxl
#export GOOGLE_API_KEY="your_key"     # PowerShell: $env:GOOGLE_API_KEY="your_key"

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PART A — Inter-annotator agreement (Cohen's kappa). No API, no GPU.
# ══════════════════════════════════════════════════════════════════════════════
import os
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

ROOT     = os.environ.get("PROJECT_ROOT", "..")
ANN_DIR  = os.environ.get("KAPPA_DIR", f"{ROOT}/outputs/annotation")
TAB      = f"{ROOT}/outputs/paper_q1/tables"
os.makedirs(TAB, exist_ok=True)
LABELS   = ["none", "abusive", "sexual", "religious", "threat"]

ann  = pd.read_excel(f"{ANN_DIR}/annotation_sample_BLANK.xlsx", sheet_name="annotate")
gold = pd.read_csv(f"{ANN_DIR}/annotation_sample_GOLD.csv", encoding="utf-8-sig")

def norm(s):
    return str(s).strip().lower()

for c in ["annotator_1_label", "annotator_2_label"]:
    ann[c] = ann[c].map(norm)

# keep only rows both annotators completed with valid labels
valid = ann["annotator_1_label"].isin(LABELS) & ann["annotator_2_label"].isin(LABELS)
n_missing = int((~valid).sum())
work = ann[valid].merge(gold[["sample_uid", "gold_label"]], on="sample_uid", how="left")
work["gold_label"] = work["gold_label"].map(norm)
print(f"annotated rows used: {len(work)}  (skipped {n_missing} blank/invalid)")
assert len(work) >= 100, "need at least ~100 completed rows for a meaningful kappa"

a1, a2, g = work["annotator_1_label"], work["annotator_2_label"], work["gold_label"]

# Cohen's kappa (annotator 1 vs 2) and each annotator vs gold
kappa_12    = cohen_kappa_score(a1, a2, labels=LABELS)
kappa_1gold = cohen_kappa_score(a1, g,  labels=LABELS)
kappa_2gold = cohen_kappa_score(a2, g,  labels=LABELS)
pct_agree   = float((a1.values == a2.values).mean())

def interpret(k):
    if k < 0.20: return "slight"
    if k < 0.40: return "fair"
    if k < 0.60: return "moderate"
    if k < 0.80: return "substantial"
    return "almost perfect"

print(f"\nCohen's kappa (annotator 1 vs 2): {kappa_12:.3f}  ({interpret(kappa_12)} agreement)")
print(f"percent agreement (1 vs 2)      : {pct_agree*100:.1f}%")
print(f"kappa annotator 1 vs gold       : {kappa_1gold:.3f}")
print(f"kappa annotator 2 vs gold       : {kappa_2gold:.3f}")

# per-class agreement between the two annotators (how often they both pick that class together)
rows = []
for lab in LABELS:
    both = int(((a1 == lab) & (a2 == lab)).sum())
    either = int(((a1 == lab) | (a2 == lab)).sum())
    jacc = both / either if either else float("nan")   # class-wise Jaccard-style agreement
    rows.append({"Class": lab, "Both agree": both, "Either used": either,
                 "Agreement": round(jacc, 3) if either else "—"})
per_class = pd.DataFrame(rows)
print("\nper-class agreement:\n", per_class.to_string(index=False))

# ---- summary table for the paper ----
summary = pd.DataFrame([
    {"Measure": "Cohen's $\\kappa$ (annotator 1 vs.\\ 2)", "Value": f"{kappa_12:.3f}", "Interpretation": interpret(kappa_12)},
    {"Measure": "Percent agreement (1 vs.\\ 2)",            "Value": f"{pct_agree*100:.1f}\\%", "Interpretation": "—"},
    {"Measure": "$\\kappa$ (annotator 1 vs.\\ consolidated label)", "Value": f"{kappa_1gold:.3f}", "Interpretation": interpret(kappa_1gold)},
    {"Measure": "$\\kappa$ (annotator 2 vs.\\ consolidated label)", "Value": f"{kappa_2gold:.3f}", "Interpretation": interpret(kappa_2gold)},
    {"Measure": "Annotated sample size",                   "Value": f"{len(work)}", "Interpretation": "stratified"},
])
summary.to_csv(f"{TAB}/table_iaa_kappa.csv", index=False)

def write_tex(df, path, caption, label, colfmt):
    cols = list(df.columns)
    L = [r"\begin{table}[!htbp]", r"\centering", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
         r"\small", f"\\begin{{tabular}}{{@{{}}{colfmt}@{{}}}}", r"\toprule",
         " & ".join(f"\\textbf{{{c}}}" for c in cols) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        L.append(" & ".join(str(r[c]) for c in cols) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    open(path, "w").write("\n".join(L))

write_tex(summary, f"{TAB}/table_iaa_kappa.tex",
          "Inter-annotator agreement on a stratified sample of "
          f"{len(work)} comments, two annotators labelling into the five classes independently.",
          "tab:iaa", "lll")
print(f"\n✅ wrote {TAB}/table_iaa_kappa.(csv|tex)")
print(f"\n>>> REPORT THIS IN THE PAPER: Cohen's kappa = {kappa_12:.3f} ({interpret(kappa_12)}), n = {len(work)} <<<")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PART B — Gemini 2.5 Flash zero-shot baseline. API (no GPU).
# ══════════════════════════════════════════════════════════════════════════════
# COST / TIME CONTROLS — read before running.
#   • N_LLM = 2000 stratified test rows keeps spend well under $1 and runs in ~15-25 min.
#     Set N_LLM = None to run the FULL 18,865-row test set (~$3-8, ~2-3 h with rate limits).
#   • Gemini 2.5 Flash is cheap; the notebook caches every response so a re-run is free
#     and an interrupted run resumes where it stopped.

LLM_MODEL   = "gemini-2.5-flash"          # Google Generative AI model id
N_LLM       = 2000                        # None = full test set
LLM_SEED    = 42
RATE_SLEEP  = 0.4                         # seconds between calls (tune to your quota)
MAX_RETRY   = 4
TEST_SPLIT  = os.environ.get("TEST_SPLIT", f"{ROOT}/data/splits/random_test.csv")
CACHE_PATH  = f"{ANN_DIR}/gemini_cache.jsonl"

# API key: set in the environment before launching Jupyter, e.g.
#   export GOOGLE_API_KEY="your_key"      (Linux/Mac)
#   $env:GOOGLE_API_KEY="your_key"        (Windows PowerShell)
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
print("Gemini config:", LLM_MODEL, "| N_LLM =", N_LLM, "| key set:", bool(GOOGLE_API_KEY))
if not GOOGLE_API_KEY:
    print("⚠ GOOGLE_API_KEY not set — Part B will not run until you set it.")


In [ ]:
# ── Run the Gemini zero-shot classifier (cached, resumable) ──
import json, time, re

def build_prompt(comment):
    return (
        "You are labelling a single Bengali social-media comment for cyberbullying. "
        "The comment may be in Bangla script or in Romanized Bangla (Latin letters). "
        "Assign exactly ONE label from this set:\n"
        "  none      = no abuse, harassment, threat, or targeted hostility\n"
        "  abusive   = general offensive/abusive language, insults, trolling, personal attacks, slurs (not specifically sexual/religious/threat)\n"
        "  sexual    = sexual or gender-based (misogynistic) harassment\n"
        "  religious = religion-directed abuse or hostility\n"
        "  threat    = explicit threat or call to violence\n"
        "If several fit, choose by priority: threat > sexual > religious > abusive > none.\n"
        "Respond with ONLY the label word, lowercase, nothing else.\n\n"
        f"Comment: {comment}\n"
        "Label:"
    )

LABELS = ["none", "abusive", "sexual", "religious", "threat"]
def parse_label(text):
    t = str(text).strip().lower()
    for lab in LABELS:                       # exact or contained
        if re.search(rf"\b{lab}\b", t):
            return lab
    return "none"                            # safe fallback for unparseable output

def load_cache(path):
    d = {}
    if os.path.exists(path):
        for line in open(path, encoding="utf-8"):
            try:
                r = json.loads(line); d[r["uid"]] = r["pred"]
            except Exception:
                pass
    return d

def run_gemini(df, text_col, uid_col):
    import google.generativeai as genai
    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel(LLM_MODEL)
    cache = load_cache(CACHE_PATH)
    fout = open(CACHE_PATH, "a", encoding="utf-8")
    preds = {}
    todo = [r for _, r in df.iterrows() if int(r[uid_col]) not in cache]
    print(f"{len(cache)} cached, {len(todo)} to call")
    for i, r in enumerate(todo, 1):
        uid = int(r[uid_col]); comment = str(r[text_col])[:2000]
        pred = None
        for attempt in range(MAX_RETRY):
            try:
                resp = model.generate_content(
                    build_prompt(comment),
                    generation_config={"temperature": 0.0, "max_output_tokens": 8})
                pred = parse_label(resp.text); break
            except Exception as e:
                wait = RATE_SLEEP * (2 ** attempt)
                if attempt == MAX_RETRY - 1:
                    print(f"  uid {uid} failed ({e}); defaulting to none"); pred = "none"
                else:
                    time.sleep(wait)
        cache[uid] = pred
        fout.write(json.dumps({"uid": uid, "pred": pred}) + "\n"); fout.flush()
        time.sleep(RATE_SLEEP)
        if i % 100 == 0: print(f"  {i}/{len(todo)}")
    fout.close()
    for _, r in df.iterrows():
        preds[int(r[uid_col])] = cache.get(int(r[uid_col]), "none")
    return preds

RAN_LLM = False
if GOOGLE_API_KEY:
    test = pd.read_csv(TEST_SPLIT, encoding="utf-8-sig").reset_index(drop=True)
    tcol = "text_clean" if "text_clean" in test.columns else "text"
    if "sample_uid" not in test.columns:
        test["sample_uid"] = test.index
    if N_LLM is not None and N_LLM < len(test):
        parts = []
        rng = np.random.default_rng(LLM_SEED)
        for _, grp in test.groupby("label5", sort=False):
            k = max(1, round(N_LLM * len(grp) / len(test)))
            parts.append(grp.sample(min(k, len(grp)), random_state=LLM_SEED))
        test = pd.concat(parts).reset_index(drop=True)
    print(f"Gemini zero-shot over {len(test)} test comments ({tcol})")
    preds = run_gemini(test, tcol, "sample_uid")
    test["llm_pred"] = test["sample_uid"].map(preds)
    RAN_LLM = True
else:
    print("skipped — set GOOGLE_API_KEY and re-run this cell")


In [ ]:
# ── Score the Gemini baseline and write the paper table ──
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef

if RAN_LLM:
    y_true = test["label5"].map(str.lower)
    y_pred = test["llm_pred"].map(str.lower)
    m = {
        "Macro-F1":    round(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0), 4),
        "Weighted-F1": round(f1_score(y_true, y_pred, labels=LABELS, average="weighted", zero_division=0), 4),
        "Accuracy":    round(accuracy_score(y_true, y_pred), 4),
        "MCC":         round(matthews_corrcoef(y_true, y_pred), 4),
    }
    print("Gemini 2.5 Flash zero-shot:", m)

    llm_tab = pd.DataFrame([
        {"System": f"Gemini 2.5 Flash (zero-shot, n={len(test)})",
         "Macro-F1": f"{m['Macro-F1']:.4f}", "Weighted-F1": f"{m['Weighted-F1']:.4f}",
         "Accuracy": f"{m['Accuracy']:.4f}", "MCC": f"{m['MCC']:.4f}"},
        {"System": "Proposed model (supervised)",
         "Macro-F1": "0.8225", "Weighted-F1": "0.8332", "Accuracy": "0.8339", "MCC": "0.7452"},
    ])
    llm_tab.to_csv(f"{TAB}/table_llm_baseline.csv", index=False)
    write_tex(llm_tab, f"{TAB}/table_llm_baseline.tex",
              "Zero-shot large-language-model baseline versus the proposed supervised model on the "
              f"5-class task ({'full 20\\% test split' if N_LLM is None else f'stratified {len(test)}-comment test sample'}).",
              "tab:llm", "lcccc")
    # per-class F1 for the discussion
    pcf = f1_score(y_true, y_pred, labels=LABELS, average=None, zero_division=0)
    print("per-class F1:", {l: round(float(f), 4) for l, f in zip(LABELS, pcf)})
    print(f"\n✅ wrote {TAB}/table_llm_baseline.(csv|tex)")
    print(f">>> REPORT: Gemini zero-shot Macro-F1 = {m['Macro-F1']:.4f} vs proposed 0.8225 <<<")
else:
    print("Part B not run — no API key. Part A (kappa) results are unaffected.")
